# MT19937 Equidistribution Analysis

This notebook creates the MT19937 Mersenne Twister generator from its
parameters, applies the standard tempering, and computes the
equidistribution properties using both the matricial and lattice methods.

In [ ]:
from regpoly.generateur import Generateur
from regpoly.transformation import Transformation
from regpoly.combinaison import Combinaison
from regpoly.component import Component
from regpoly.analyses.equidistribution_test import (
    EquidistributionTest, METHOD_MATRICIAL, METHOD_DUALLATTICE
)

## 1. Create the MT19937 Generator

Parameters from Matsumoto & Nishimura (1998):
- $(w, n, m, r) = (32, 624, 397, 31)$
- $a = \texttt{0x9908B0DF}$
- Period: $2^{19937} - 1$
- State size: $k = w \times n - (w - r) = 32 \times 624 - 31 = 19937$

In [ ]:
# MT19937 generator parameters
mt_params = {
    "w": 32,      # word size
    "r": 624,     # number of registers (n in the paper)
    "m": 397,     # middle word
    "p": 31,      # separation point (r in the paper)
    "a": 0x9908B0DF  # twist matrix coefficient
}

L = 32  # output resolution

gen = Generateur.create("MersenneTwister", L, **mt_params)

print(f"Generator: {gen.name()}")
print(f"k = {gen.k} (state bits)")
print(f"L = {gen.L} (output bits)")

## 2. Characteristic Polynomial

In [ ]:
char_poly = gen.char_poly()
hw = bin(char_poly._val).count('1') + 1  # +1 for the leading z^k term
print(f"Characteristic polynomial degree: {gen.k}")
print(f"Hamming weight: {hw}")

## 3. Create the Tempering Transformation

MT19937 tempering (Matsumoto-Nishimura type II):
- $y \leftarrow y \oplus (y \gg 11)$
- $y \leftarrow y \oplus ((y \ll 7) \wedge \texttt{0x9D2C5680})$
- $y \leftarrow y \oplus ((y \ll 15) \wedge \texttt{0xEFC60000})$
- $y \leftarrow y \oplus (y \gg 18)$

In [ ]:
temper_params = {
    "w": 32,
    "eta": 7,           # s in the paper (left shift)
    "mu": 15,           # t in the paper (left shift)
    "u": 11,            # first right shift
    "l": 18,            # last right shift
    "b": 0x9D2C5680,   # mask for eta shift
    "c": 0xEFC60000    # mask for mu shift
}

trans = Transformation("tempMK2", temper_params)

print(f"Tempering: {trans._cpp_trans.display_str()}")

## 4. Build the Combinaison

A `Combinaison` wraps one or more generator components for testing.

In [ ]:
comb = Combinaison(J=1, Lmax=L)
comb.components[0].add_gen(gen)
comb.components[0].add_trans(trans)
comb.reset()

print(f"k_g = {comb.k_g}")
print(f"L   = {comb.L}")
print(f"J   = {comb.J} component(s)")

## 5. Equidistribution Test (Lattice Method)

The lattice method uses Lenstra's algorithm on the dual polynomial lattice.
For MT19937, the paper reports a total dimension defect of **6750**.

In [ ]:
INT_MAX = 2**31 - 1

test_lattice = EquidistributionTest(
    L=L,
    delta=[INT_MAX] * (L + 1),
    mse=INT_MAX,
    method=METHOD_DUALLATTICE
)

result_lattice = test_lattice.run(comb)

print("Lattice method results:")
print(f"  Dimension gaps sum (Psi_12) = {result_lattice.se}")
print()
result_lattice.display_table(comb, 'l')

## 6. Detailed Results

For each resolution $l = 1, \ldots, 32$, we show:
- $t_l^* = \lfloor k / l \rfloor$: theoretical maximum dimension
- $t_l$: achieved dimension
- $\Delta_l = t_l^* - t_l$: dimension gap

In [ ]:
k = comb.k_g
print(f"{'l':>3}  {'t*':>6}  {'t_l':>6}  {'gap':>4}")
print("-" * 25)
total = 0
for l in range(1, L + 1):
    t_star = k // l
    gap = result_lattice.ecart[l]
    t_l = t_star - gap
    total += gap
    marker = "" if gap == 0 else f"  <-- gap = {gap}"
    print(f"{l:3d}  {t_star:6d}  {t_l:6d}  {gap:4d}{marker}")
print("-" * 25)
print(f"Total dimension defect = {total}")

## 7. Without Tempering

For comparison, the equidistribution without tempering is much worse.
The raw MT19937 (TGFSR) has large dimension gaps at most resolutions.

In [ ]:
# Build a Combinaison without tempering
comb_raw = Combinaison(J=1, Lmax=L)
comb_raw.components[0].add_gen(gen)
# No tempering added
comb_raw.reset()

result_raw = test_lattice.run(comb_raw)

print("Without tempering:")
print(f"  Dimension gaps sum (Psi_12) = {result_raw.se}")
print()
result_raw.display_table(comb_raw, 'l')

## 8. Comparison: Tempered vs Untempered

In [ ]:
print(f"{'l':>3}  {'gap (tempered)':>15}  {'gap (raw)':>10}  {'improvement':>12}")
print("-" * 48)
for l in range(1, L + 1):
    gt = result_lattice.ecart[l]
    gr = result_raw.ecart[l]
    imp = gr - gt
    print(f"{l:3d}  {gt:15d}  {gr:10d}  {imp:12d}")
print("-" * 48)
print(f"Total:  {sum(result_lattice.ecart[l] for l in range(1, L+1)):13d}  "
      f"{sum(result_raw.ecart[l] for l in range(1, L+1)):10d}")